# Pre-Arrears Strategy — Component: Pre-Arrears Qualifiers (SAS Step 1)

**Source**: `Pre-Arrears.txt`, lines 49–214 (`%macro ss; ... %mend ss; %ss;`)

**Scope note**: The source program contains 10 sequential steps (qualifier
extraction, previous-performance lookback, DDR import, bureau risk
qualifiers, contact-history checks, merge/cut-offs, BTL exposure dedup,
dialler/mailing/monitoring exports). Per the conversion brief, **only one
component is converted here**: **Step 1 — Pre-Arrears Qualifiers**, the
SQL Server pass-through query that builds the monthly qualifying
population snapshot (SAS table `SS&yyyymm`). This is the entry point every
downstream step in the strategy depends on.

No customer selection, segmentation, threshold, or filter logic has been
changed. Where a literal has been lifted into a named parameter, its
**default value is identical to the SAS literal** — see the risk list
below and the `TODO: VERIFY` markers for anything that could not be
confirmed from the source alone.


## 1. What the SAS component does

```
%macro ss;
  %let yyyymm  = intnx(month,&rundate,0)      -> reporting period, YYYYMM
  %let RSFrom  = intnx(month,&rundate,0,'b')  -> 1st day of that month, YYYYMMDD
  %let RSTo    = intnx(month,&rundate,0,'e')  -> last day of that month, YYYYMMDD

  proc sql;
    connect to odbc (P_DW, trusted connection);
    create table SS&yyyymm as
      select * from connection to odbc ( <T-SQL query> );
    disconnect from odbc;
  quit;
%mend ss;
%ss;
```

`&rundate` is supplied externally as the **first day of the prior month**
(e.g. `01Jul2026` when the strategy is run in August 2026). The whole
component is a *pass-through* query: SAS does none of the filtering
itself — it sends the T-SQL below to SQL Server and materializes whatever
comes back, in the order returned, into a work table named `SS<yyyymm>`.

### The T-SQL logic

**`CTE_DRS`** — accounts currently under an active **Debt Respite Scheme**
(DRS breathing space): scheme end date is today or later, and the scheme
has not been marked ceased.

**`CTE_LPA`** — accounts under an active **Law of Property Act (LPA)
receiver appointment**: pool `SBS00001`, status Live/Possession (or an
end date in/after the current month), classification Residential/BTL
Commercial/Pure Commercial, receiver appointment date populated, not a
deleted record.

**Main `SELECT`** — one row per (account, snapshot month) from
`factMortgageAccountMonthSS`, joined to dimension tables, restricted to:

| Rule | Detail |
|---|---|
| Company scope | Skipton Building Society, Amber Homeloans Ltd, North Yorkshire Mortgages |
| Pool exclusion | `AccountPool <> 'DRC00001'` |
| Snapshot month | `dimMonth.CalendarMonthYear = &yyyymm` |
| Account subtype | `Primary` only |
| Classification | `Residential` only |
| Status | `Live` |
| Arrears stage | **Contractual** arrears (SBS) or **Standard** arrears (AHL/NYM) `< 1` month — i.e. pre-arrears |
| Product exclusion | SBS deed accounts + Holmesdale deed/fair-value products |
| Balance floor | `CapitalBalanceLive > 1000` |
| Arrangement | `ArrangementTypeCode < 0` (no arrangement in place) |
| Recent completions | account start date more than 3 months before the reporting month |
| Remaining term | `RemainingTerm > 12` months |
| Deceased holders | any account holder (relationship type `19`, pool `SBS00001`) with a recorded date of death is excluded |
| DRS | accounts in `CTE_DRS` excluded |
| LPA | accounts in `CTE_LPA` excluded |
| Manual exclusion | account `132876009` excluded (ad-hoc Credit Management request) |

Derived output columns:
- `AccountNumber` — the AHL/NYM-migrated allocated number if one exists, else the raw account number.
- `Lender` — AHL / NYM / SBS, derived from company name / origin.
- `Snapshot` — `YYYYMM` integer from `dimMonth`.
- `Segment` — `BTL` if sub-population code is `BTL`, else `Resi`.
- `BalSS`, `MIASS`, `ArrBalSS`, `ArrBelow1MIA` — balance and arrears figures, sourced from the Standard or Contractual fields depending on lender, defaulted to `0` (not NULL) when not `>0`.
- `BH_Score` — behavioural score, floored; **`999` when no scorecard match** (left join).
- `RS_Ind` — `1` if a Right-to-Sell letter (`RED001`/`RED01E`/`RED01F`) was issued to the account within the reporting month, else `0`; deduplicated via `RANK() OVER (... ORDER BY dc_creat_o_dt DESC, time_o DESC, oid DESC)`, keeping rank 1.

Result rows are **ordered** by the effective (allocated) account number,
and that order is preserved in the resulting SAS table because
`create table ... as select * from connection to odbc(...)` materializes
rows in the order the driver returns them.


## 2. Semantic risks (SAS → Python/SQL)

1. **Missing vs NULL.** The T-SQL already neutralises most business
   fields with `CASE WHEN ... IS NULL THEN <default> ELSE ...` (e.g.
   `BH_Score`, `MIASS`, `ArrBalSS`). Any *other* left-joined column that
   is genuinely `NULL` (e.g. from `dp`, `dsp` if a code lookup is
   missing) becomes SAS numeric/character **missing** on the SAS side and
   will silently behave as `0`/blank in later arithmetic or concatenation
   steps. In pandas this becomes `NaN`/`None` and does **not** silently
   coerce to zero — any downstream arithmetic must replicate the SAS
   coercion explicitly rather than relying on `NaN` propagation.
2. **Dates.** `CONVERT(DATE, GETDATE())` truncates time-of-day; SAS
   `intnx(month, &rundate, 0, 'b'/'e')` returns month-begin/month-end
   dates for the **same month as `&rundate`** (not the following month).
   The Python replica must reproduce month-boundary arithmetic exactly,
   including the December→January rollover and leap-year February.
3. **Character padding / truncation.** SAS assigns each ODBC character
   column a *fixed* length inferred from source metadata (e.g.
   `AccountOrigin`, `CompanyName`, `AccountNumber` once cast from
   numeric-as-text `Allocated_AccountNumber`). Values longer than that
   inferred width are silently truncated in SAS; pandas/`pyodbc` will not
   truncate. This must be checked against the target schema's declared
   column widths before load — **TODO: VERIFY target schema max lengths
   for `AccountOrigin`, `Lender`, and `AccountNumber`.**
4. **Numeric precision / rounding.** `FLOOR(bs.S3Score)` in SQL Server
   floors toward negative infinity, same as Python's `math.floor` — do
   **not** use `int()` truncation, which floors toward zero and would
   differ for a negative score. `CapitalBalanceLive`/arrears amounts are
   SQL Server `decimal`/`money`; converting to Python `float` risks
   binary rounding drift on comparisons — use `Decimal` or compare with
   an explicit tolerance if these figures are re-derived rather than
   passed through.
5. **Sort order.** The SAS table's row order is not guaranteed by the
   SQL engine merely by having an `ORDER BY` inside the connection block combined
   with how SAS materializes pass-through results — it depends on the ODBC
   driver returning rows in the order SQL Server produced them, which SQL
   Server itself does not contractually guarantee without the `ORDER BY`
   being the *outermost* operation (it is, here). The Python
   implementation must apply the equivalent explicit `sort_values` after
   fetch, since neither PEP 249 DB-API cursors nor pandas guarantee
   fetch order otherwise.
6. **Duplicate handling.** No `first.`/`last.` BY-group logic exists in
   this component (it is pure SQL, no DATA step). The only de-duplication
   is the `RANK()` on `ADM98` for `RS_Ind`; ties on
   `(dc_creat_o_dt, time_o, oid)` all receive the same rank under `RANK()`
   (unlike `ROW_NUMBER()`), so more than one row could satisfy
   `LatestRS=1` for the same account if all three tie-break columns are
   identical. **TODO: VERIFY whether `oid` (a surrogate key) can ever tie
   in practice** — if not, this is moot; if it can, the join `ON
   da.AccountNumber = rs.accnbr_o` would fan out and inflate row count
   for that account.
7. **Retained values.** Not applicable — no `retain` statement exists in
   this component (SQL pass-through only, no DATA step accumulation).
8. **`&rundate` contract.** SAS performs no validation that `&rundate` is
   the first day of a month; a mis-set parameter silently shifts every
   downstream filter. The Python version adds explicit validation
   (**this is a defensive addition, not a behaviour change**, since a
   correctly-supplied `rundate` produces identical output either way).
9. **Hardcoded exclusion `132876009`.** Lifted into a named, versioned
   config parameter with an identical default so behaviour is unchanged.
   **TODO: VERIFY with Credit Management whether this exclusion is still
   required** — it is preserved as-is here regardless.
10. **Trusted (Windows-integrated) authentication.** The SAS `libname`
    uses `trusted connection=yes` (no stored credential). The Python
    connector reproduces this via `Trusted_Connection=yes` on the ODBC
    connection string rather than introducing a stored password.


In [ ]:
"""
pre_arrears_qualifiers.py
Production module: Pre-Arrears Strategy — Step 1: Pre-Arrears Qualifiers.

Converted 1:1 from Pre-Arrears.txt lines 49-214 (SAS macro %ss).
Behaviour-preserving conversion: no selection, threshold, or join logic
has been altered. See markdown cells above for the full risk log.
"""

import logging
import math
import os
import sys
from dataclasses import dataclass, field
from datetime import date
from typing import List, Optional

import pandas as pd

logger = logging.getLogger("pre_arrears.qualifiers")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s")
    )
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)


In [ ]:
class QualifierExtractionError(Exception):
    """Raised when the Step 1 extraction cannot be trusted to proceed."""


class QualifierValidationError(QualifierExtractionError):
    """Raised when the extracted qualifier population fails validation."""


In [ ]:
@dataclass(frozen=True)
class PreArrearsQualifierConfig:
    """
    All tunable values for the Pre-Arrears Qualifiers component.

    Every default below is IDENTICAL to the literal hardcoded in the SAS
    source. Parameterising does not change behaviour unless a caller
    deliberately overrides a value.
    """

    # --- Run parameter -----------------------------------------------------
    rundate: date  # must be the 1st day of a calendar month (SAS &rundate contract)

    # --- Connection (Windows-integrated / trusted auth, matches SAS libname) -
    db_server: str = "SBSPSQLVS101\\MISPSQL01"
    db_database: str = "P_DW"
    db_driver: str = "ODBC Driver 17 for SQL Server"
    connect_timeout_seconds: int = 30
    query_timeout_seconds: int = 600
    max_retries: int = 3
    retry_backoff_seconds: float = 5.0

    # --- Business thresholds (defaults == SAS literals) ---------------------
    capital_balance_threshold: float = 1000.0          # fma.CapitalBalanceLive > 1000
    min_months_since_completion: int = 3                # DATEDIFF(...) > 3
    min_remaining_term_months: int = 12                 # fma.RemainingTerm > 12
    bh_score_missing_default: int = 999                  # CASE WHEN NULL THEN 999

    company_scope: List[str] = field(default_factory=lambda: [
        "Skipton Building Society", "Amber Homeloans Ltd", "North Yorkshire Mortgages",
    ])
    excluded_account_pool: str = "DRC00001"
    excluded_product_codes: List[str] = field(default_factory=lambda: [
        "MIN01", "HOM10", "HM120", "HM121", "HM122", "HM123", "HM124", "HM125",
    ])
    deceased_check_pool: str = "SBS00001"
    deceased_relationship_type_code: str = "19"
    lpa_check_pool: str = "SBS00001"
    lpa_classifications: List[str] = field(default_factory=lambda: [
        "Residential", "BTL Commercial", "Pure Commercial",
    ])
    rs_letter_codes: List[str] = field(default_factory=lambda: [
        "RED001", "RED01E", "RED01F",
    ])
    # TODO: VERIFY this ad-hoc exclusion is still required by Credit Management.
    manual_excluded_accounts: List[int] = field(default_factory=lambda: [132876009])

    def __post_init__(self):
        if self.rundate.day != 1:
            raise ValueError(
                f"rundate must be the first day of a month (SAS &rundate contract); "
                f"got {self.rundate.isoformat()}"
            )


In [ ]:
def get_reporting_period(rundate: date) -> int:
    """Equivalent of SAS: intnx(month,&rundate,0) formatted yymmn6. -> int YYYYMM."""
    return rundate.year * 100 + rundate.month


def get_month_bounds(rundate: date) -> "tuple[int, int]":
    """
    Equivalent of SAS &RSFrom / &RSTo:
        intnx(month,&rundate,0,'b') -> first day of rundate's month
        intnx(month,&rundate,0,'e') -> last day of rundate's month
    Returned as YYYYMMDD integers, matching yymmddn8. formatting.
    """
    import calendar

    first_day = date(rundate.year, rundate.month, 1)
    last_day_num = calendar.monthrange(rundate.year, rundate.month)[1]
    last_day = date(rundate.year, rundate.month, last_day_num)
    to_yyyymmdd = lambda d: d.year * 10000 + d.month * 100 + d.day
    return to_yyyymmdd(first_day), to_yyyymmdd(last_day)


def floor_or_default(score: Optional[float], default: int) -> int:
    """Equivalent of: CASE WHEN bs.S3Score IS NULL THEN 999 ELSE FLOOR(bs.S3Score) END."""
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return default
    return math.floor(score)  # math.floor matches SQL Server FLOOR() for negatives too


In [ ]:
def _in_clause(n: int) -> str:
    """Build a parameterised `(?, ?, ...)` clause of length n. n=0 -> impossible-match clause."""
    if n == 0:
        return "(SELECT NULL WHERE 1 = 0)"
    return "(" + ", ".join(["?"] * n) + ")"


def build_qualifiers_query(cfg: PreArrearsQualifierConfig) -> "tuple[str, list]":
    """
    Rebuilds the Step 1 T-SQL pass-through query with bound parameters in
    place of SAS macro-variable text substitution and inline literals.
    Business logic, joins, and filter conditions are unchanged from the
    SAS source (Pre-Arrears.txt lines 64-208).
    """
    yyyymm = get_reporting_period(cfg.rundate)
    rs_from, rs_to = get_month_bounds(cfg.rundate)

    params: list = []

    company_in = _in_clause(len(cfg.company_scope))
    params += list(cfg.company_scope)

    product_ex = _in_clause(len(cfg.excluded_product_codes))
    params += list(cfg.excluded_product_codes)

    lpa_class_in = _in_clause(len(cfg.lpa_classifications))
    params += list(cfg.lpa_classifications)

    rs_codes_in = _in_clause(len(cfg.rs_letter_codes))
    params += list(cfg.rs_letter_codes)

    manual_excl_in = _in_clause(len(cfg.manual_excluded_accounts))
    params += list(cfg.manual_excluded_accounts)

    query = f"""
WITH CTE_DRS AS
(
SELECT  m99.primacno_o AS PrimaryAccount
        , m98.accnbr_o AS AccountNumber
        , da.AccountOrigin
        , CONVERT(DATE, drs._timestamp) AS DRS_DateRecorded
        , CONVERT(DATE, drs.startDate) AS DRS_StartDate
        , CONVERT(DATE, drs.endDate) AS DRS_EndDate
        , drs.arrearsAmount AS DRS_Arrears
FROM    P_ODS.SKIPCORE.DatDebtRespiteSchemeDetails drs
INNER JOIN  P_ODS.SKIPCORE.MOR98 m98
    ON  drs.myAccount = m98.oid
    AND m98.pool_id_o = '{cfg.lpa_check_pool}'
INNER JOIN  P_ODS.SKIPCORE.MOR99 m99
    ON  m98.myMOR99 = m99.oid
    AND m98.pool_id_o = m99.pool_id_o
    AND m98.primacno_o = m99.primacno_o
INNER JOIN  P_DW.dbo.dimAccount da
    ON  m98.oid = da.BK_Account
    AND m98.pool_id_o = da.AccountPool
    AND m98.accnbr_o = da.AccountNumber
    AND da.AccountType = 10
WHERE   CONVERT(DATE, drs.endDate) >= CONVERT(DATE, GETDATE())
AND     drs.ceased = 0
),
CTE_LPA AS
(
SELECT  rcv.primacno_o AS PrimaryAccount
FROM    P_ODS.SKIPCORE.RCV98 rcv
INNER JOIN  P_DW.dbo.dimAccount da
    ON  rcv.pool_id_o = da.AccountPool
    AND rcv.primacno_o = da.AccountNumber
WHERE   rcv.pool_id_o = '{cfg.lpa_check_pool}'
AND     (da.CurrentAccountStatus IN ('Live', 'Possession')
         OR da.AccountEndDate >= DATEADD(MONTH, DATEDIFF(MONTH, 0, GETDATE()), 0))
AND     da.CurrentAccountClassification IN {lpa_class_in}
AND     rcv.dc_recvr_o_dt IS NOT NULL
AND     rcv._deleted = 0
)
SELECT  CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
        , CASE WHEN dc.CompanyName = 'Amber Homeloans Ltd' OR da.AccountOrigin = 'Amber Homeloans Ltd' THEN 'AHL'
               WHEN dc.CompanyName = 'North Yorkshire Mortgages' OR da.AccountOrigin = 'North Yorkshire Mortgages Ltd' THEN 'NYM'
               ELSE 'SBS' END AS Lender
        , dm.CalendarYear * 100 + dm.CalendarMonthNumber AS Snapshot
        , CASE WHEN dsp.SubPopulationCode = 'BTL' THEN 'BTL' ELSE 'Resi' END AS Segment
        , fma.CapitalBalanceLive AS BalSS
        , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsInMonths > 0 THEN fma.StandardArrearsInMonths
               WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsInMonths > 0 THEN fma.ContractualArrearsInMonths
               ELSE 0 END AS MIASS
        , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsAmount > 0 THEN fma.StandardArrearsAmount
               WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsAmount > 0 THEN fma.ContractualArrearsAmount
               ELSE 0 END AS ArrBalSS
        , CASE WHEN da.AccountStartDate IS NOT NULL AND dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsInMonths > 0 AND fma.StandardArrearsInMonths < 1 THEN 1
               WHEN da.AccountStartDate IS NOT NULL AND dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsInMonths > 0 AND fma.ContractualArrearsInMonths < 1 THEN 1
               ELSE 0 END AS ArrBelow1MIA
        , CASE WHEN bs.S3Score IS NULL THEN ? ELSE FLOOR(bs.S3Score) END AS BH_Score
        , CASE WHEN rs.LatestRS IS NULL THEN 0 ELSE 1 END AS RS_Ind
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN  P_DW.dbo.dimAccount da ON fma.FK_Account = da.PK_Account
INNER JOIN  P_DW.dbo.dimCompany dc ON fma.FK_Company = dc.PK_Company
LEFT JOIN   P_ODS.reference.AHLNYMMigratedAccounts lu
    ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
INNER JOIN  P_DW.dbo.dimProduct dp ON fma.FK_CurrentMortgageProduct = dp.PK_Product
INNER JOIN  P_DW.dbo.dimMonth dm ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN  P_DW.dbo.dimSubPopulation dsp ON fma.FK_SubPopulation = dsp.PK_SubPopulation
INNER JOIN  P_DW.dbo.dimAccountStates das ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN  P_DW.dbo.dimArrangement dar ON fma.FK_ArrangementType = dar.PK_ArrangementType
LEFT JOIN   P_Modelling.COREMODEL.vwBehaviouralScorecard bs
    ON  dm.CalendarMonthYear = CONVERT(VARCHAR, bs.Period)
    AND (CASE WHEN dc.CompanyName = 'Amber Homeloans Ltd' THEN 'AHL'
              WHEN dc.CompanyName = 'North Yorkshire Mortgages' THEN 'NYM'
              ELSE 'SBS' END) = bs.CompanyCode
    AND da.AccountNumber = bs.PrimaryAccount
LEFT JOIN   (
                SELECT * FROM (
                    SELECT  a98.pool_id_o, a98.accnbr_o, a98.lettcode_o, a98.dc_creat_o_dt
                            , RANK() OVER (PARTITION BY a98.accnbr_o ORDER BY a98.dc_creat_o_dt DESC, a98.time_o DESC, a98.oid DESC) AS LatestRS
                    FROM    P_ODS.SKIPCORE.ADM98 a98
                    WHERE   a98.pool_id_o = '{cfg.lpa_check_pool}'
                    AND     a98.lettcode_o IN {rs_codes_in}
                    AND     a98.dc_creat_o BETWEEN ? AND ?
                ) a
                WHERE a.LatestRS = 1
            ) rs
    ON  da.AccountPool = rs.pool_id_o AND da.AccountNumber = rs.accnbr_o
WHERE   dc.CompanyName IN {company_in}
AND     da.AccountPool <> ?
AND     dm.CalendarMonthYear = CONVERT(VARCHAR, ?)
AND     da.AccountSubType = 'Primary'
AND     da.CurrentAccountClassification = 'Residential'
AND     das.AccountStatus IN ('Live')
AND     ((dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsInMonths < 1)
         OR (dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsInMonths < 1))
AND     dp.ProductCode NOT IN {product_ex}
AND     fma.CapitalBalanceLive > ?
AND     dar.ArrangementTypeCode < 0
AND     DATEDIFF(MONTH, da.AccountStartDate, DATEADD(MONTH, 0, dm.ReportingMonth)) > ?
AND     fma.RemainingTerm > ?
AND     da.AccountNumber NOT IN (
            SELECT b.AccountNumber
            FROM (
                SELECT a.AccountNumber, SUM(a.DeathFlag) AS DeathFlag
                FROM (
                    SELECT  da.AccountNumber, dc.CustomerNumber
                            , CASE WHEN dc.CustomerDateofDeath IS NOT NULL THEN 1 ELSE 0 END AS DeathFlag
                    FROM    P_DW.dbo.factMortgageAccount fma
                    INNER JOIN  P_DW.dbo.dimAccount da ON fma.FK_Account = da.PK_Account
                    INNER JOIN  P_DW.dbo.bridgeAccountCustomer bac
                        ON  da.PK_Account = bac.FK_Account AND bac.relationshipTypeCode = ?
                    INNER JOIN  P_DW.dbo.dimCustomer dc ON bac.FK_Customer = dc.PK_Customer
                    WHERE   da.AccountPool = ?
                    AND     da.AccountSubType = 'Primary'
                    AND     da.CurrentAccountStatus IN ('Live')
                    AND     da.CurrentAccountClassification = 'Residential'
                ) a
                GROUP BY a.AccountNumber
            ) b
            WHERE b.DeathFlag > 0
        )
AND     da.AccountNumber NOT IN (SELECT PrimaryAccount FROM CTE_DRS)
AND     da.AccountNumber NOT IN (SELECT PrimaryAccount FROM CTE_LPA)
AND     da.AccountNumber NOT IN {manual_excl_in}
ORDER BY (CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END)
"""

    # Parameter order must exactly match the '?' placeholders left-to-right
    # as they appear in the SQL text above. NOTE: CTE_LPA is defined (and
    # therefore textually appears) BEFORE the main SELECT, so its
    # placeholder(s) must be ordered first, ahead of BH_Score etc.
    return query, _build_ordered_params(cfg, rs_from, rs_to, yyyymm)


def _build_ordered_params(cfg: PreArrearsQualifierConfig, rs_from: int, rs_to: int, yyyymm: int) -> list:
    """Authoritative parameter list, in the exact left-to-right order the '?'
    placeholders appear in `build_qualifiers_query`'s SQL text:
    CTE_LPA classifications -> BH_Score default -> RS letter codes ->
    RS date window -> company scope -> excluded pool -> reporting period ->
    excluded products -> balance/term thresholds -> deceased-check params ->
    manual exclusions."""
    return (
        list(cfg.lpa_classifications)
        + [cfg.bh_score_missing_default]
        + list(cfg.rs_letter_codes)
        + [rs_from, rs_to]
        + list(cfg.company_scope)
        + [cfg.excluded_account_pool, yyyymm]
        + list(cfg.excluded_product_codes)
        + [cfg.capital_balance_threshold, cfg.min_months_since_completion, cfg.min_remaining_term_months]
        + [cfg.deceased_relationship_type_code, cfg.deceased_check_pool]
        + list(cfg.manual_excluded_accounts)
    )


In [ ]:
def retry(times: int, backoff_seconds: float):
    """Simple retry decorator for transient DB connectivity failures."""
    import functools
    import time

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as exc:  # noqa: BLE001 - deliberately broad, re-raised below
                    last_exc = exc
                    logger.warning(
                        "Attempt %s/%s failed for %s: %s", attempt, times, fn.__name__, exc
                    )
                    if attempt < times:
                        time.sleep(backoff_seconds * attempt)
            raise QualifierExtractionError(
                f"{fn.__name__} failed after {times} attempts"
            ) from last_exc
        return wrapper
    return decorator


class DataWarehouseConnector:
    """
    Thin wrapper around pyodbc reproducing the SAS libname's trusted
    (Windows-integrated) authentication. No credentials are stored or
    passed in code, matching `trusted connection=yes` in the SAS source.
    """

    def __init__(self, cfg: PreArrearsQualifierConfig):
        self.cfg = cfg
        self._conn = None

    def _connection_string(self) -> str:
        return (
            f"DRIVER={{{self.cfg.db_driver}}};"
            f"SERVER={self.cfg.db_server};"
            f"DATABASE={self.cfg.db_database};"
            f"Trusted_Connection=yes;"
            f"Connect Timeout={self.cfg.connect_timeout_seconds};"
        )

    @retry(times=3, backoff_seconds=5.0)
    def connect(self):
        import pyodbc  # imported lazily so the notebook can be reviewed without the driver installed

        logger.info("Connecting to %s/%s", self.cfg.db_server, self.cfg.db_database)
        self._conn = pyodbc.connect(self._connection_string(), timeout=self.cfg.connect_timeout_seconds)
        return self._conn

    def __enter__(self):
        return self.connect()

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._conn is not None:
            self._conn.close()
            logger.info("Connection closed")


In [ ]:
REQUIRED_COLUMNS = [
    "AccountNumber", "Lender", "Snapshot", "Segment", "BalSS",
    "MIASS", "ArrBalSS", "ArrBelow1MIA", "BH_Score", "RS_Ind",
]

# Columns that are guaranteed non-null by construction (INNER JOIN chain /
# explicit CASE...ELSE defaulting in the SAS source) - a null here signals
# an upstream data-quality break, not a legitimate business outcome.
NON_NULLABLE_COLUMNS = [
    "AccountNumber", "Lender", "Snapshot", "Segment",
    "BalSS", "MIASS", "ArrBalSS", "ArrBelow1MIA", "BH_Score", "RS_Ind",
]


def validate_qualifiers(df: pd.DataFrame, cfg: PreArrearsQualifierConfig) -> None:
    """
    Data-quality gate for the extracted qualifier population.
    Raises QualifierValidationError on any check failure; never silently
    mutates the population (no rows dropped, no values altered here).
    """
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise QualifierValidationError(f"Missing expected columns: {missing_cols}")

    if df.empty:
        logger.warning(
            "Qualifier extraction returned 0 rows for period %s. "
            "This can be legitimate (e.g. no eligible accounts) but should "
            "be confirmed before the strategy proceeds.",
            get_reporting_period(cfg.rundate),
        )
        return

    for col in NON_NULLABLE_COLUMNS:
        null_count = int(df[col].isna().sum())
        if null_count > 0:
            raise QualifierValidationError(
                f"Column '{col}' has {null_count} unexpected null(s); "
                f"source SQL defaults this field and should never be null."
            )

    dup_count = int(df["AccountNumber"].duplicated().sum())
    if dup_count > 0:
        # The SAS source has no explicit dedup on AccountNumber in this step;
        # a duplicate here most likely indicates a RANK() tie in the RS_Ind
        # lookup fanning out the join (see risk #6). Surface it, do not
        # silently collapse it, since silently deduping would be a
        # behaviour change.
        logger.error(
            "%s duplicate AccountNumber value(s) detected — likely a RANK() "
            "tie-break fan-out in the RS_Ind lookup. TODO: VERIFY against "
            "source data before this population is used downstream.",
            dup_count,
        )
        raise QualifierValidationError(
            f"{dup_count} duplicate AccountNumber rows detected; see log for detail."
        )

    below_floor = int((df["BalSS"] <= cfg.capital_balance_threshold).sum())
    if below_floor > 0:
        raise QualifierValidationError(
            f"{below_floor} row(s) at/below the capital balance threshold "
            f"({cfg.capital_balance_threshold}) leaked through the SQL filter."
        )

    logger.info(
        "Validation passed: %s rows, %s unique accounts, %s Resi / %s BTL",
        len(df), df["AccountNumber"].nunique(),
        int((df["Segment"] == "Resi").sum()), int((df["Segment"] == "BTL").sum()),
    )


In [ ]:
def extract_pre_arrears_qualifiers(cfg: PreArrearsQualifierConfig) -> pd.DataFrame:
    """
    Main entry point. Equivalent to SAS `%ss;` — builds and executes the
    Step 1 qualifier query and returns the population as a DataFrame
    equivalent to SAS work table `SS<yyyymm>`.
    """
    yyyymm = get_reporting_period(cfg.rundate)
    logger.info("Starting Pre-Arrears Qualifiers extraction for period %s", yyyymm)

    query, params = build_qualifiers_query(cfg)

    with DataWarehouseConnector(cfg) as conn:
        conn.timeout = cfg.query_timeout_seconds
        try:
            df = pd.read_sql(query, conn, params=params)
        except Exception as exc:  # noqa: BLE001
            raise QualifierExtractionError(
                f"Qualifier query execution failed for period {yyyymm}"
            ) from exc

    # Preserve the SAS ORDER BY explicitly rather than trusting fetch order
    # (see risk #5). Sort key matches the SQL ORDER BY expression exactly.
    df = df.sort_values("AccountNumber", kind="stable").reset_index(drop=True)

    validate_qualifiers(df, cfg)

    logger.info("Completed Pre-Arrears Qualifiers extraction: %s rows", len(df))
    return df


## 3. Example invocation

Not executed automatically in this notebook (no live warehouse
connection in this environment). Toggle `RUN_LIVE_EXTRACTION = True` and
supply real connection details / driver to run against the actual
warehouse.


In [ ]:
RUN_LIVE_EXTRACTION = False  # flip to True in a deployment environment with DB access

if RUN_LIVE_EXTRACTION:
    config = PreArrearsQualifierConfig(rundate=date(2026, 7, 1))
    ss_dataframe = extract_pre_arrears_qualifiers(config)
    display(ss_dataframe.head())
else:
    logger.info("RUN_LIVE_EXTRACTION is False — skipping live database call.")


## 4. Unit tests

In [ ]:
import unittest
from datetime import date as _date
from unittest.mock import MagicMock, patch


class TestReportingPeriodAndBounds(unittest.TestCase):
    def test_reporting_period_basic(self):
        self.assertEqual(get_reporting_period(_date(2026, 7, 1)), 202607)

    def test_reporting_period_december(self):
        self.assertEqual(get_reporting_period(_date(2026, 12, 1)), 202612)

    def test_month_bounds_31_day_month(self):
        rs_from, rs_to = get_month_bounds(_date(2026, 7, 1))
        self.assertEqual(rs_from, 20260701)
        self.assertEqual(rs_to, 20260731)

    def test_month_bounds_30_day_month(self):
        rs_from, rs_to = get_month_bounds(_date(2026, 4, 1))
        self.assertEqual(rs_to, 20260430)

    def test_month_bounds_leap_year_february(self):
        rs_from, rs_to = get_month_bounds(_date(2028, 2, 1))  # 2028 is a leap year
        self.assertEqual(rs_to, 20280229)

    def test_month_bounds_non_leap_year_february(self):
        _, rs_to = get_month_bounds(_date(2026, 2, 1))
        self.assertEqual(rs_to, 20260228)

    def test_month_bounds_december_year_end(self):
        rs_from, rs_to = get_month_bounds(_date(2026, 12, 1))
        self.assertEqual(rs_from, 20261201)
        self.assertEqual(rs_to, 20261231)


class TestConfigValidation(unittest.TestCase):
    def test_rejects_non_first_of_month_rundate(self):
        with self.assertRaises(ValueError):
            PreArrearsQualifierConfig(rundate=_date(2026, 7, 15))

    def test_accepts_first_of_month_rundate(self):
        cfg = PreArrearsQualifierConfig(rundate=_date(2026, 7, 1))
        self.assertEqual(cfg.rundate, _date(2026, 7, 1))

    def test_defaults_match_sas_literals(self):
        cfg = PreArrearsQualifierConfig(rundate=_date(2026, 7, 1))
        self.assertEqual(cfg.capital_balance_threshold, 1000.0)
        self.assertEqual(cfg.min_months_since_completion, 3)
        self.assertEqual(cfg.min_remaining_term_months, 12)
        self.assertEqual(cfg.bh_score_missing_default, 999)
        self.assertEqual(cfg.manual_excluded_accounts, [132876009])
        self.assertEqual(cfg.excluded_account_pool, "DRC00001")


class TestFloorOrDefault(unittest.TestCase):
    def test_returns_default_when_none(self):
        self.assertEqual(floor_or_default(None, 999), 999)

    def test_returns_default_when_nan(self):
        self.assertEqual(floor_or_default(float("nan"), 999), 999)

    def test_floors_positive_score(self):
        self.assertEqual(floor_or_default(612.9, 999), 612)

    def test_floors_negative_score_toward_negative_infinity(self):
        # SQL Server FLOOR(-0.5) = -1, matching math.floor (NOT int() truncation)
        self.assertEqual(floor_or_default(-0.5, 999), -1)

    def test_exact_integer_score_unchanged(self):
        self.assertEqual(floor_or_default(600.0, 999), 600)


class TestQueryBuilder(unittest.TestCase):
    def setUp(self):
        self.cfg = PreArrearsQualifierConfig(rundate=_date(2026, 7, 1))

    def test_placeholder_count_matches_param_count(self):
        query, params = build_qualifiers_query(self.cfg)
        placeholder_count = query.count("?")
        self.assertEqual(placeholder_count, len(params))

    def test_manual_exclusion_present_in_params(self):
        _, params = build_qualifiers_query(self.cfg)
        self.assertIn(132876009, params)

    def test_company_scope_present_in_params(self):
        _, params = build_qualifiers_query(self.cfg)
        for company in self.cfg.company_scope:
            self.assertIn(company, params)

    def test_in_clause_empty_list_never_matches(self):
        clause = _in_clause(0)
        self.assertIn("1 = 0", clause)

    def test_in_clause_placeholder_count(self):
        clause = _in_clause(3)
        self.assertEqual(clause.count("?"), 3)


class TestValidation(unittest.TestCase):
    def setUp(self):
        self.cfg = PreArrearsQualifierConfig(rundate=_date(2026, 7, 1))
        self.valid_row = {
            "AccountNumber": 100001, "Lender": "SBS", "Snapshot": 202607,
            "Segment": "Resi", "BalSS": 150000.0, "MIASS": 0, "ArrBalSS": 0.0,
            "ArrBelow1MIA": 0, "BH_Score": 612, "RS_Ind": 0,
        }

    def test_valid_frame_passes(self):
        df = pd.DataFrame([self.valid_row])
        validate_qualifiers(df, self.cfg)  # should not raise

    def test_empty_frame_does_not_raise(self):
        df = pd.DataFrame(columns=REQUIRED_COLUMNS)
        validate_qualifiers(df, self.cfg)  # should not raise, only warns

    def test_missing_required_column_raises(self):
        df = pd.DataFrame([{k: v for k, v in self.valid_row.items() if k != "BH_Score"}])
        with self.assertRaises(QualifierValidationError):
            validate_qualifiers(df, self.cfg)

    def test_null_in_non_nullable_column_raises(self):
        row = dict(self.valid_row)
        row["ArrBalSS"] = None
        df = pd.DataFrame([row])
        with self.assertRaises(QualifierValidationError):
            validate_qualifiers(df, self.cfg)

    def test_duplicate_account_number_raises(self):
        df = pd.DataFrame([self.valid_row, self.valid_row])
        with self.assertRaises(QualifierValidationError):
            validate_qualifiers(df, self.cfg)

    def test_balance_at_threshold_raises(self):
        row = dict(self.valid_row)
        row["BalSS"] = self.cfg.capital_balance_threshold  # not strictly greater than
        df = pd.DataFrame([row])
        with self.assertRaises(QualifierValidationError):
            validate_qualifiers(df, self.cfg)


suite = unittest.TestLoader().loadTestsFromModule(sys.modules[__name__])
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed — see output above."


## 5. Source-to-target reconciliation

Run after both the legacy SAS output (`SS&yyyymm`, exported to a
staging table or file) and the Python output are available for the same
`rundate`. Reconciliation must show **zero** differences before the
Python output replaces the SAS output for any live run.


In [ ]:
RECONCILIATION_QUERIES = {
    "control_totals": """
        SELECT
            'legacy_sas'  AS source, COUNT(*) AS row_count,
            COUNT(DISTINCT AccountNumber) AS distinct_accounts,
            SUM(CAST(BalSS AS DECIMAL(18,2)))    AS sum_balss,
            SUM(CAST(ArrBalSS AS DECIMAL(18,2))) AS sum_arrbalss
        FROM staging.SS_{yyyymm}_legacy
        UNION ALL
        SELECT
            'python' AS source, COUNT(*) AS row_count,
            COUNT(DISTINCT AccountNumber) AS distinct_accounts,
            SUM(CAST(BalSS AS DECIMAL(18,2)))    AS sum_balss,
            SUM(CAST(ArrBalSS AS DECIMAL(18,2))) AS sum_arrbalss
        FROM staging.SS_{yyyymm}_python;
    """,

    "accounts_in_legacy_not_python": """
        SELECT l.AccountNumber
        FROM staging.SS_{yyyymm}_legacy l
        LEFT JOIN staging.SS_{yyyymm}_python p
            ON l.AccountNumber = p.AccountNumber
        WHERE p.AccountNumber IS NULL;
    """,

    "accounts_in_python_not_legacy": """
        SELECT p.AccountNumber
        FROM staging.SS_{yyyymm}_python p
        LEFT JOIN staging.SS_{yyyymm}_legacy l
            ON p.AccountNumber = l.AccountNumber
        WHERE l.AccountNumber IS NULL;
    """,

    "field_level_mismatches": """
        SELECT
            l.AccountNumber,
            l.Lender AS legacy_Lender,   p.Lender AS python_Lender,
            l.Segment AS legacy_Segment, p.Segment AS python_Segment,
            l.BalSS AS legacy_BalSS,     p.BalSS AS python_BalSS,
            l.ArrBalSS AS legacy_ArrBalSS, p.ArrBalSS AS python_ArrBalSS,
            l.BH_Score AS legacy_BH_Score, p.BH_Score AS python_BH_Score,
            l.RS_Ind AS legacy_RS_Ind,   p.RS_Ind AS python_RS_Ind
        FROM staging.SS_{yyyymm}_legacy l
        INNER JOIN staging.SS_{yyyymm}_python p
            ON l.AccountNumber = p.AccountNumber
        WHERE  l.Lender        <> p.Lender
            OR l.Segment       <> p.Segment
            OR l.BalSS         <> p.BalSS
            OR l.ArrBalSS      <> p.ArrBalSS
            OR l.BH_Score      <> p.BH_Score
            OR l.RS_Ind        <> p.RS_Ind;
    """,
}


def render_reconciliation_queries(yyyymm: int) -> dict:
    """Fill the {yyyymm} placeholder for a given reporting period."""
    return {name: sql.format(yyyymm=yyyymm) for name, sql in RECONCILIATION_QUERIES.items()}


In [ ]:
def reconcile_dataframes(legacy_df: pd.DataFrame, python_df: pd.DataFrame) -> dict:
    """
    In-memory equivalent of the SQL reconciliation above, for use when both
    outputs are already loaded as DataFrames (e.g. in a test/UAT run).
    Returns a dict of discrepancy DataFrames; every value should be empty
    for a clean reconciliation.
    """
    compare_cols = ["Lender", "Segment", "BalSS", "ArrBalSS", "BH_Score", "RS_Ind"]

    merged = legacy_df.merge(
        python_df, on="AccountNumber", how="outer",
        suffixes=("_legacy", "_python"), indicator=True,
    )

    only_in_legacy = merged.loc[merged["_merge"] == "left_only", ["AccountNumber"]]
    only_in_python = merged.loc[merged["_merge"] == "right_only", ["AccountNumber"]]

    both = merged.loc[merged["_merge"] == "both"].copy()
    mismatch_mask = pd.Series(False, index=both.index)
    for col in compare_cols:
        mismatch_mask |= both[f"{col}_legacy"] != both[f"{col}_python"]
    field_mismatches = both.loc[mismatch_mask]

    control_totals = pd.DataFrame([
        {
            "source": "legacy", "row_count": len(legacy_df),
            "distinct_accounts": legacy_df["AccountNumber"].nunique(),
            "sum_balss": legacy_df["BalSS"].sum(),
            "sum_arrbalss": legacy_df["ArrBalSS"].sum(),
        },
        {
            "source": "python", "row_count": len(python_df),
            "distinct_accounts": python_df["AccountNumber"].nunique(),
            "sum_balss": python_df["BalSS"].sum(),
            "sum_arrbalss": python_df["ArrBalSS"].sum(),
        },
    ])

    return {
        "control_totals": control_totals,
        "accounts_in_legacy_not_python": only_in_legacy,
        "accounts_in_python_not_legacy": only_in_python,
        "field_level_mismatches": field_mismatches,
    }


## 6. Consolidated `TODO: VERIFY` list

- **Target schema column widths** for `AccountOrigin`, `Lender`,
  `AccountNumber` — needed to confirm no truncation risk when this
  DataFrame is loaded into a fixed-width target table.
- **`RANK()` tie-break stability** on `ADM98` (`dc_creat_o_dt`, `time_o`,
  `oid`) for `RS_Ind` — confirm `oid` cannot tie in practice, or the join
  can fan out and this component's duplicate-check will (correctly) start
  rejecting output.
- **Manual exclusion `132876009`** — confirm with Credit Management
  whether this ad-hoc exclusion is still required; preserved unchanged
  here either way.
- **`ODBC Driver 17 for SQL Server`** — confirm the exact driver name
  installed in the target production environment; SAS's `libname odbc`
  resolves this implicitly via the system DSN/driver manager, which has
  no direct Python equivalent to introspect.
- **Decimal vs float precision** for `CapitalBalanceLive` and the arrears
  amount fields — confirm whether downstream consumers require
  `Decimal`-exact reconciliation or float is acceptable at this stage.
